## Loading Data

In [1]:
import tensorflow as tf
from tensorflow import keras

In [2]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

housing = fetch_california_housing()
x_train_full,x_test,y_train_full,y_test = train_test_split(housing.data,housing.target,random_state=33)
x_train,x_val,y_train,y_val = train_test_split(x_train_full,y_train_full,random_state=33)

scal = StandardScaler()
x_train = scal.fit_transform(x_train)
x_test = scal.transform(x_test)
x_val = scal.transform(x_val)

In [3]:
model = keras.models.Sequential([
    keras.layers.Dense(30,activation='relu',input_shape=x_train.shape[1:]),
    keras.layers.Dense(1)

])
model.compile(loss='mean_squared_error',optimizer='sgd')
history = model.fit(x_train,y_train,epochs=5,
                    validation_data=(x_val,y_val))

test = model.evaluate(x_test,y_test)

c:\Users\augusto\.vscode\Data Sci and Ml\Data\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/5
363/363 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.9863 - val_loss: 0.5674
Epoch 2/5
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.5176 - val_loss: 0.4908
Epoch 3/5
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.4769 - val_loss: 0.5223
Epoch 4/5
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.4756 - val_loss: 0.4613
Epoch 5/5
363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.4520 - val_loss: 0.4617
162/162 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.4327


In [4]:
print(test)

0.4326854944229126


In [5]:
#pred
x_new = x_test[:4]
y_pred_new = model.predict(x_new)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 114ms/step


In [6]:
print(y_pred_new)

[[3.4677503]
 [1.084551 ]
 [3.189414 ]
 [1.333776 ]]


we can create a dynamic model with a subclass

In [7]:
class Ann_model(keras.Model):
    def __init__(self, units=30,activation='relu', **kwargs):
        super().__init__( **kwargs)

        self.hidden1 = keras.layers.Dense(units=units,activation=activation)
        self.hidden2 = keras.layers.Dense(units=units,activation=activation)
        self.main_output = keras.layers.Dense(1)
        self.aux_output = keras.layers.Dense(1)


    def call_model(self,inputs):
        input_a,input_b = inputs
        hidden1 = self.hidden1(input_b)
        hidden2 = self.hidden2(hidden1)
        concat = keras.layers.concatenate([input_a,hidden2])
        main_output = self.main_output(concat)
        aux_output = self.aux_output(hidden2)

        return main_output,aux_output
    
model1 = Ann_model()

Saving progress:

In [8]:
model.save('keras_housing1.h5')
model2 = keras.models.load_model('keras_housing1.h5')

In [9]:
checkpoint = keras.callbacks.ModelCheckpoint('keras_housing1.h5',save_best_only=True)
history2 = model.fit(x_train,y_train,epochs=3,callbacks=[checkpoint])

Epoch 1/3
360/363 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.4347

c:\Users\augusto\.vscode\Data Sci and Ml\Data\.venv\Lib\site-packages\keras\src\callbacks\model_checkpoint.py:329: UserWarning: Can save best model only with val_loss available.
  if self._should_save_model(epoch, batch, logs, filepath):


363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4421
Epoch 2/3
360/363 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.4399

363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4541
Epoch 3/3
353/363 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.4365

363/363 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4417


or even early stopping:

In [10]:
early_stopping_model2 = keras.callbacks.EarlyStopping(patience=5,restore_best_weights=True)

We can also finetuning a ann:

In [11]:
def custom_model(n_neurons=20,n_hidden=1,learning_rate=3e-3,input_shape=[5]):
    model = keras.models.Sequential()
    model.add(keras.layers.Inputlayer(input_shape=input_shape))
    for n_hidden in range(n_hidden):
        model.add(keras.layers.Dense(n_neurons,activation='relu'))
    model.add(model.layers.Dense(1))
    optimizer = keras.optimizers.SGD(lr=learning_rate)
    model.compile(loss='mse',optimizer=optimizer)


In [17]:
keras_reg = keras.wrappers.scikit_learn.KerasRegressor(custom_model)

AttributeError: module 'keras.wrappers' has no attribute 'scikit_learn'

now its a scikitlearnmodel that we can use gridsearch